## **MoE**

| Expert | Provider | Modelo | Self-label |
|--------|----------|--------|------------|
| Expert 1 | Groq | Llama 3.3 70B | Meta |
| Expert 2 | Groq | Gemma 2 9B | Google |
| Expert 3 | Mistral | Mistral Small | — |

In [13]:
import pandas as pd
import numpy as np
import time
import json
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from groq import Groq
from mistralai.client import Mistral

sns.set_style('whitegrid')
LABELS = ['Anthropic', 'Google', 'Human', 'Meta', 'OpenAI']

### **1. Keys + Clientes**

Usa as mesmas chaves gratuitas da aula (Groq + Mistral).

In [14]:
GROQ_API_KEY    = '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'  
MISTRAL_API_KEY = 'PEnTbEqdVs8tPFcnAATFSaw3tHkhVH26'  

GROQ_MODEL_LLAMA = 'llama-3.3-70b-versatile'  
GROQ_MODEL_GEMMA = 'gemma2-9b-it'              
MISTRAL_MODEL    = 'mistral-small-latest'      

groq_client = Groq(api_key=GROQ_API_KEY)
print('Groq key loaded:', bool(GROQ_API_KEY))
print('Mistral key loaded:', bool(MISTRAL_API_KEY))

Groq key loaded: True
Mistral key loaded: True


### **2. Dados**

Carrega os mesmos CSVs do notebook original.

In [15]:
df_support = pd.read_csv('../database/dataset-subm1-labels.csv', sep=';')
df_support.columns = df_support.columns.str.strip().str.lower()

df_test = pd.read_csv('../database/dataset-samples.csv', sep=';')
df_test.columns = df_test.columns.str.strip().str.lower()

N_PER_CLASS = 10
support_set = pd.concat([
    group.sample(min(N_PER_CLASS, len(group)), random_state=42)
    for _, group in df_support.groupby('label')
]).reset_index(drop=True)

print(f'Few-shot: {len(support_set)} | Teste: {len(df_test)}')

Few-shot: 50 | Teste: 125


### **3. Prompt**

Mesmo prompt do notebook original — funciona bem com qualquer LLM.

In [16]:
TRUNC_EXAMPLES = 800
TRUNC_QUERY    = 1200

STYLE_HINTS = """
KEY STYLISTIC DIFFERENCES:
- Human: Wikipedia-style neutrality, shorter paragraphs, factual, occasional passive voice, citations.
- Anthropic (Claude): hedging ("it's worth noting", "generally speaking"), balanced/nuanced, avoids bold claims.
- OpenAI (GPT): assertive, lists/bullet points, "Moreover", "Furthermore", comprehensive.
- Google (Gemini): concise, direct, technical precision, bold markers, shorter.
- Meta (Llama): less polished, repetitive, simpler vocabulary, subtle inconsistencies.
"""

def build_prompt(text, support_examples):
    examples_block = ''
    for _, row in support_examples.iterrows():
        ex_text = row['text'][:TRUNC_EXAMPLES]
        examples_block += f'Text: {ex_text}\nCategory: {row["label"]}\n\n'
    return f"""You are an expert linguist specialized in detecting AI-generated text.
Classify the following text into exactly ONE of these categories:
- Human (written by a human, e.g. from Wikipedia)
- Anthropic (generated by Claude)
- Google (generated by Gemini)
- Meta (generated by Llama)
- OpenAI (generated by GPT)

{STYLE_HINTS}

Here are {len(support_examples)} labeled examples:

{examples_block}
Now classify this text. Output ONLY the category name, nothing else.

Text: {text[:TRUNC_QUERY]}
Category:"""

### **4. Expert Wrappers (Modelos Gratuitos)**

3 experts, todos gratuitos:
- **Llama 3.3 70B** via Groq (modelo da Meta)
- **Gemma 2 9B** via Groq (modelo da Google)
- **Mistral Small** via Mistral API

In [17]:
def ask_groq(prompt, model):
    """Wrapper genérico para Groq (funciona com Llama, Gemma, Mixtral, etc.)"""
    try:
        response = groq_client.chat.completions.create(
            messages=[{'role': 'user', 'content': prompt}],
            model=model,
            max_tokens=50,
            temperature=0.0,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f'Error: {e}'


def ask_llama(prompt):
    """Expert 1: Llama 3.3 70B (Meta) via Groq"""
    return ask_groq(prompt, GROQ_MODEL_LLAMA)


def ask_gemma(prompt):
    """Expert 2: Gemma 2 9B (Google) via Groq"""
    return ask_groq(prompt, GROQ_MODEL_GEMMA)


def ask_mistral(prompt):
    """Expert 3: Mistral Small via Mistral API"""
    if not MISTRAL_API_KEY:
        return 'Error: No API Key'
    try:
        with Mistral(api_key=MISTRAL_API_KEY) as client:
            response = client.chat.complete(
                model=MISTRAL_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                max_tokens=50,
                temperature=0.0,
            )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f'Error: {e}'


def normalize_prediction(raw):
    """Extrai o label válido da resposta do modelo."""
    if not raw or raw.startswith('Error'):
        return None
    raw_lower = raw.strip().lower()
    for label in LABELS:
        if label.lower() in raw_lower:
            return label
    return None

### **5. Teste de Conectividade**

In [18]:
print('Testando APIs...')
test_prompt = 'Respond with only one word: OK'

r = ask_llama(test_prompt)
print(f'  {"✅" if not r.startswith("Error") else "❌"} Llama 3.3 70B (Groq): "{r}"')

r = ask_gemma(test_prompt)
print(f'  {"✅" if not r.startswith("Error") else "❌"} Gemma 2 9B (Groq):    "{r}"')

r = ask_mistral(test_prompt)
print(f'  {"✅" if not r.startswith("Error") else "❌"} Mistral Small:         "{r}"')

Testando APIs...
  ✅ Llama 3.3 70B (Groq): "OK"
  ❌ Gemma 2 9B (Groq):    "Error: Error code: 400 - {'error': {'message': 'The model `gemma2-9b-it` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}"
  ✅ Mistral Small:         "Okay"


### **6. MoE Gating**

Mesma lógica do original, adaptada para os novos experts:
- **Self-penalty**: Llama (Meta) a votar "Meta" → penalizado. Gemma (Google) a votar "Google" → penalizado.
- **Mistral** não é nenhuma das 5 labels → voto neutro, sem penalização.
- **Cross-consensus**: se 2+ experts concordam (excluindo self-votes), confiamos nisso.

In [19]:
# Cada expert sabe de que "empresa" vem → penalizar self-votes
SELF_LABEL = {
    'llama':   'Meta',    # Llama é da Meta
    'gemma':   'Google',  # Gemma é da Google
    'mistral': None,      # Mistral não é nenhum dos 5 labels
}

SELF_PENALTY    = 0.3   # peso quando vota na própria empresa
CROSS_WEIGHT    = 1.0   # peso normal
OPENAI_BOOST    = 1.2   # boost para OpenAI (nenhum expert é OpenAI)

EXPERT_MULTIPLIER = {
    'llama':   1.0,   # modelo maior, mais confiança
    'gemma':   0.75,  # modelo menor, ligeiramente menos
    'mistral': 0.9,   # bom modelo, neutro
}


def moe_vote(expert_preds):
    """Weighted voting com self-penalty e cross-consensus."""
    votes = Counter()
    for expert_name, pred in expert_preds.items():
        if pred is None:
            continue
        # Determinar peso
        if pred == SELF_LABEL.get(expert_name):
            weight = SELF_PENALTY   # penalizar self-vote
        elif pred == 'OpenAI':
            weight = OPENAI_BOOST   # nenhum expert é OpenAI → boost
        else:
            weight = CROSS_WEIGHT
        weight *= EXPERT_MULTIPLIER.get(expert_name, 1.0)
        votes[pred] += weight

    if not votes:
        return None, {}

    # Cross-consensus: se 2+ experts concordam (excluindo self-votes)
    cross_votes = Counter()
    for expert_name, pred in expert_preds.items():
        if pred and pred != SELF_LABEL.get(expert_name):
            cross_votes[pred] += 1
    for label, count in cross_votes.most_common():
        if count >= 2:
            return label, {'method': 'cross_consensus', 'votes': dict(votes)}

    return votes.most_common(1)[0][0], {'method': 'weighted_vote', 'votes': dict(votes)}

### **7. Correr o MoE**

⚠️ **Nota sobre rate limits**: A Groq tem limites no tier gratuito (~30 req/min). O `sleep_between` ajuda a não exceder. Se tiveres erros 429, aumenta o sleep.

In [20]:
def run_moe(df_data, support_df, sleep_between=3.0):
    results = []
    for idx, row in tqdm(df_data.iterrows(), total=len(df_data), desc='MoE'):
        prompt = build_prompt(row['text'], support_df)

        # Expert 1: Llama 3.3 70B
        raw_llama = ask_llama(prompt)
        pred_llama = normalize_prediction(raw_llama)
        time.sleep(1)  # pequena pausa entre chamadas Groq

        # Expert 2: Gemma 2 9B
        raw_gemma = ask_gemma(prompt)
        pred_gemma = normalize_prediction(raw_gemma)
        time.sleep(1)

        # Expert 3: Mistral Small
        raw_mistral = ask_mistral(prompt)
        pred_mistral = normalize_prediction(raw_mistral)

        # Votar
        expert_preds = {
            'llama':   pred_llama,
            'gemma':   pred_gemma,
            'mistral': pred_mistral,
        }
        final_pred, vote_info = moe_vote(expert_preds)

        results.append({
            'id': row.get('id', idx),
            'true_label': row.get('label', None),
            'llama_raw': raw_llama, 'llama_pred': pred_llama,
            'gemma_raw': raw_gemma, 'gemma_pred': pred_gemma,
            'mistral_raw': raw_mistral, 'mistral_pred': pred_mistral,
            'moe_pred': final_pred,
            'vote_info': json.dumps(vote_info),
        })

        if sleep_between > 0:
            time.sleep(sleep_between)

    return pd.DataFrame(results)


print('🚀 MoE com modelos gratuitos')
df_results = run_moe(df_test, support_set, sleep_between=3.0)
df_results.to_csv('moe_results_free.csv', index=False, sep=';')
print('✅ Done!')

🚀 MoE com modelos gratuitos


MoE:   0%|          | 0/125 [00:00<?, ?it/s]

KeyboardInterrupt: 

### **8. Avaliação**

In [ ]:
def evaluate_model(df_res, pred_col, title=''):
    valid = df_res.dropna(subset=[pred_col, 'true_label'])
    preds, golds = valid[pred_col].tolist(), valid['true_label'].tolist()
    if not preds:
        return 0.0
    acc = sum(p == g for p, g in zip(preds, golds)) / len(preds)
    print(f'\n{"="*60}\n{title} — Accuracy: {acc:.2%} ({len(preds)}/{len(df_res)})\n{"="*60}')
    print(classification_report(golds, preds, labels=LABELS, zero_division=0))
    cm = confusion_matrix(golds, preds, labels=LABELS)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, ax=ax)
    ax.set_xlabel('Previsão')
    ax.set_ylabel('Real')
    ax.set_title(f'{title} — {acc:.2%}')
    plt.tight_layout()
    plt.show()
    return acc


accs = {}
accs['Llama 3.3 70B']  = evaluate_model(df_results, 'llama_pred',   'Llama 3.3 70B (Groq)')
accs['Gemma 2 9B']     = evaluate_model(df_results, 'gemma_pred',   'Gemma 2 9B (Groq)')
accs['Mistral Small']  = evaluate_model(df_results, 'mistral_pred', 'Mistral Small')
accs['MoE Ensemble']   = evaluate_model(df_results, 'moe_pred',     'MoE Ensemble (Free)')

print('\nRESUMO:')
for name, acc in sorted(accs.items(), key=lambda x: -x[1]):
    print(f'  {name:18s} {acc:.2%}  {"█" * int(acc * 40)}')

### **9. Análise de Erros**

In [ ]:
errors = df_results[df_results['moe_pred'] != df_results['true_label']]
print(f'Erros: {len(errors)}/{len(df_results)}\n')
for _, row in errors.iterrows():
    vi = json.loads(row['vote_info']) if isinstance(row['vote_info'], str) else row['vote_info']
    print(f'[{row["id"]}] Real={row["true_label"]} → MoE={row["moe_pred"]}  |  '
          f'Llama={row["llama_pred"]}  Gemma={row["gemma_pred"]}  Mistral={row["mistral_pred"]}')